# drone-burst-sr sur Google Colab (GPU gratuit)

Ce notebook installe **drone-burst-sr** avec l'algorithme *Handheld Multi-Frame Super-Resolution*
(implémentation IPOL, Numba + CUDA) et lance l'interface web sur un GPU NVIDIA de Colab.

**Avant de commencer** : menu *Exécution → Modifier le type d'exécution → GPU T4*, puis exécutez
les cellules dans l'ordre.

In [ ]:
!nvidia-smi

## 1. Installation

Trois façons d'obtenir le code, choisies par les variables ci-dessous :

- **Dépôt public** : rien à faire.
- **Dépôt privé, sans jeton (recommandé)** : sur GitHub, ouvrez la branche et cliquez
  *Code → Download ZIP*, puis déposez le ZIP dans votre Google Drive et indiquez son chemin dans
  `LOCAL_SOURCE`. Un dossier contenant le dépôt convient aussi.
- **Dépôt privé, avec jeton** : ajoutez un jeton GitHub en lecture seule dans les *Secrets*
  Colab (icône 🔑) sous le nom `GITHUB_TOKEN`.

> La connexion « GitHub » de Colab permet seulement d'**ouvrir et d'enregistrer des notebooks**
> depuis vos dépôts privés. Elle ne donne aucun accès aux commandes `pip`/`git` exécutées sur la
> machine Colab : c'est pourquoi un dépôt privé demande l'une des deux dernières options.

In [ ]:
REPO = "sylvaingranges-pixel/superpixel"   # remplacez par <compte>/drone-burst-sr si le dépôt est renommé
BRANCH = "ccr-cc33588e-5l482s"             # ou "main"
LOCAL_SOURCE = ""  # ex. "/content/drive/MyDrive/superpixel-ccr-cc33588e-5l482s.zip" (aucun jeton nécessaire)

# Toujours repartir de zéro, pour ne jamais garder une ancienne version.
!pip uninstall -y -q drone-burst-sr

if LOCAL_SOURCE:
    if LOCAL_SOURCE.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if LOCAL_SOURCE.endswith(".zip"):
        !pip install -q "drone-burst-sr[gpu] @ file://{LOCAL_SOURCE}"
    else:
        !rm -rf /content/drone_burst_sr_src && cp -r "{LOCAL_SOURCE}" /content/drone_burst_sr_src
        !pip install -q "/content/drone_burst_sr_src[gpu]"
else:
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    auth = f"{token}@" if token else ""
    url = f"git+https://{auth}github.com/{REPO}@{BRANCH}"
    !pip install -q "drone-burst-sr[gpu] @ {url}"

from importlib.metadata import version
print("drone-burst-sr installé :", version("drone-burst-sr"), "(attendu : 0.2.0 ou plus)")


### Compatibilité CUDA

L'algorithme amont installe la variante **CUDA 13** de Numba, qui demande un pilote NVIDIA récent
(≥ 580). Si le pilote de Colab est plus ancien (`CUDA Version: 12.x` dans `nvidia-smi`), la cellule
suivante bascule automatiquement sur CUDA 12.

In [ ]:
import re, subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*(\d+)", out)
cuda_major = int(m.group(1)) if m else 0
print("CUDA du pilote :", cuda_major)
if cuda_major and cuda_major < 13:
    !pip install -q "numba-cuda[cu12]" "cuda-bindings>=12.9,<13" "cuda-toolkit[cccl,cudart,nvcc,nvrtc]==12.*" "nvidia-nvjitlink-cu12"
    print("Variante CUDA 12 installée.")

In [ ]:
from drone_burst_sr.gpu import detect_gpu
print(detect_gpu().describe())

## 2. Auto-test (≈ 2-3 min, dont ~1 min de compilation des kernels CUDA)

Rafale synthétique (10 images, décalages sous-pixel connus, bruit ≈ ISO 800) fusionnée à ×2,
comparée zone par zone à une seule image dématricée par rawpy :

- `super_resolution` : lignes plus fines que ce qu'une image seule peut résoudre ;
- `aplats` : bruit ;
- `basses_freq` : contrôle, ne doit pas se dégrader.

La fusion est refaite sans robustesse pour vérifier que celle-ci ne rejette pas tout.
Une planche de comparaison s'affiche en dessous.

In [ ]:
!python -m drone_burst_sr.selftest --out-dir /content
import os
from IPython.display import Image, display
if os.path.exists("/content/selftest_comparaison.png"):
    display(Image("/content/selftest_comparaison.png"))
else:
    print("Pas de planche : la version installée est ancienne. Réexécutez la cellule d'installation.")


## 3. Où mettre les photos ?

Les photos doivent être lisibles par la machine Colab. Trois façons de faire :

| Méthode | Quand l'utiliser |
|---|---|
| **Google Drive** (recommandé) | Copiez la rafale dans un dossier de votre Drive (application Drive pour ordinateur ou drive.google.com), exécutez la cellule ci-dessous, puis indiquez le dossier dans l'interface, par ex. `/content/drive/MyDrive/rafales/rafale1`. Envoi une seule fois, réutilisable, résultats gardés. |
| **Glisser-déposer dans l'interface** | Petites rafales (≈ 10 DNG 12 MP de drone). Plus lent : les fichiers passent par le tunnel Gradio. |
| **Panneau Fichiers de Colab** (icône 📁) | Dépannage. Les fichiers sont **effacés** à la fin de la session Colab. |

In [ ]:
# Facultatif : rend votre Google Drive visible sous /content/drive/MyDrive
from google.colab import drive
drive.mount("/content/drive")

## 3a. Interface web

Ouvrez le lien public `https://….gradio.live` affiché ci-dessous (valable 72 h) dans votre
navigateur Windows. Déposez les DNG, ou indiquez un dossier Google Drive (voir 3b).
La cellule reste « en cours » tant que l'interface tourne : interrompez-la pour l'arrêter.

> Les fichiers transitent par le tunnel Gradio. Pour de grosses rafales (48 MP ≈ 1 Go), il est
> plus rapide de les déposer sur Google Drive et d'indiquer le dossier dans l'interface.

In [ ]:
from drone_burst_sr.app import build_app
demo = build_app()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=True)

## 3b. Ligne de commande avec Google Drive

Copiez la rafale dans un dossier de votre Drive (par ex. `MyDrive/rafales/rafale1`).
Les résultats sont écrits à côté, dans le Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")  # sans effet si déjà monté
INPUT = "/content/drive/MyDrive/rafales/rafale1"
OUT = INPUT + "_fusion_x2.tif"
!python -m drone_burst_sr --input "{INPUT}" --scale 2 --out "{OUT}" --single-out "{INPUT}_image_seule_x2.tif"

In [ ]:
# Téléchargement direct (sinon récupérez les fichiers dans Google Drive)
from google.colab import files
files.download(OUT.replace(".tif", ".jpg"))